# Estimate population (exposure)

This short how-to guides you through the steps to create a timeseries of the population using GHS Population data and save it as a csv file. The final csv file can be used as, e.g., exposure data for the drought risk estimation.

We use two datasets from the Global Human Settlement (GHS) project:
- **GHS-POP**: Historical and projected population (1975-2030), [GHS Population Grid](https://human-settlement.emergency.copernicus.eu/ghs_pop2023.php)
- **GHS-WUP**: Urban population projections (1975-2100), [GHS WUP Population](https://human-settlement.emergency.copernicus.eu/ghs_pop2023.php)

The data is downloaded manually through the following webpage: [https://human-settlement.emergency.copernicus.eu/](https://human-settlement.emergency.copernicus.eu/). 

We download the global data for all years available, but at 30 arcsec resolution, and in the WGS84 (or EPSG: 4326) projection for easier handling of the data afterwards. The images below show the web interface to download the datasets and detail the settings used. 

Download GHS population             |  Download GHS WUP population
:-------------------------:|:-------------------------:
![Image](../img/GHS_POP_download.png "Web interface to download GHS population data.")  |  ![Image](../img/GHS_POP_WUP_download.png "Web interface to download GHS WUP population data.")

More information: [GHS Population Grid](https://human-settlement.emergency.copernicus.eu/ghs_pop2023.php)

In [94]:
admin_id = "EL64"  # Example admin ID for Central Greece

## Manual download of GHS Population data

Download the GHS population data manually from the [GHS download page](https://ghsl.jrc.ec.europa.eu/download.php):

**GHS-POP (1975-2030)**: Download files named `GHS_POP_E{YYYY}_GLOBE_R2023A_4326_30ss_V1_0.tif` for years 1975, 1980, 1985, 1990, 1995, 2000, 2005, 2010, 2015, 2020, 2025, 2030.

**GHS-WUP (1975-2100)**: Download files named `GHS_WUP_POP_E{YYYY}_GLOBE_R2023A_4326_30ss_V1_0.tif` for years from 1975 to 2100 in 5-year intervals.

Once downloaded, move these files into the following subdirectory structure:
- `./data/ghs_population/GHS_POP/` for GHS-POP files
- `./data/ghs_population/GHS_WUP/` for GHS-WUP files




## Load libraries and set up region

In [95]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import regionmask
from pathlib import Path
import rioxarray as rxr
import os
import re

# Set up data directories
data_dir = Path("./data")
ghs_pop_dir = data_dir / "ghs_population" / "GHS_POP"
ghs_wup_dir = data_dir / "ghs_population" / "GHS_WUP"

print(f"Working directory: {os.getcwd()}")
print(f"GHS-POP directory: {ghs_pop_dir}")
print(f"GHS-WUP directory: {ghs_wup_dir}")

Working directory: /etc/ecmwf/nfs/dh2_home_a/nejk/code/drought_exposure
GHS-POP directory: data/ghs_population/GHS_POP
GHS-WUP directory: data/ghs_population/GHS_WUP


## Load NUTS region shapefile and define region mask

In [92]:
# Read NUTS shapefiles
regions_dir = data_dir / 'regions'
nuts_shp = regions_dir / 'NUTS_RG_20M_2024_4326' / 'NUTS_RG_20M_2024_4326.shp'
nuts_gdf = gpd.read_file(nuts_shp)

# Select the region of interest
sel_gdf = nuts_gdf[nuts_gdf['NUTS_ID'] == admin_id]
print(f"Found {admin_id} region: {sel_gdf['NUTS_NAME'].values[0]}")
print(f"Bounding box: {sel_gdf.geometry.total_bounds}")

# Create a regionmask from the admin region geometry
admin_mask = regionmask.from_geopandas(sel_gdf, names='NUTS_ID')

Found EL64 region: Στερεά Ελλάδα
Bounding box: [21.39637798 37.98898161 24.67199242 39.27219519]


## Read GHS-POP data (1975-2030)

In [93]:
# List all GHS-POP tif files
ghs_pop_files = sorted(list(ghs_pop_dir.glob("GHS_POP_E*_GLOBE_R2023A_4326_30ss_V1_0.tif")))
print(f"Found {len(ghs_pop_files)} GHS-POP files:")
for f in ghs_pop_files:
    print(f"  {f.name}")

Found 0 GHS-POP files:


In [ ]:
# Read all GHS-POP files and extract years
datasets_pop = []
years_pop = []

for tif_file in ghs_pop_files:
    # Read using rioxarray
    ds = rxr.open_rasterio(tif_file)
    
    # Extract year from filename (looking for pattern E{YYYY})
    year_match = re.search(r'E(\d{4})', tif_file.name)
    if year_match:
        year = int(year_match.group(1))
        years_pop.append(year)
        print(f"Processing year: {year}")
    else:
        print(f"Warning: Could not extract year from {tif_file.name}")
        continue
    
    datasets_pop.append(ds)

# Sort datasets and years together by year
sorted_pairs = sorted(zip(years_pop, datasets_pop), key=lambda x: x[0])
years_pop, datasets_pop = zip(*sorted_pairs)

# Concatenate along new time dimension
pop_ds = xr.concat(datasets_pop, dim="time")
pop_ds = pop_ds.rename("population")
pop_ds = pop_ds.squeeze("band", drop=True)

# Assign time based on extracted years
pop_ds["time"] = pd.to_datetime([f"{year}-01-01" for year in years_pop])
pop_ds = pop_ds.assign_coords(time=pop_ds.time)

# Replace potential fill values with NaN (common fill values: -200, -99999, etc.)
pop_ds = pop_ds.where(pop_ds >= 0)

print(f"\nGHS-POP dataset shape: {pop_ds.shape}")
print(f"Years available: {years_pop}")
pop_ds

## Read GHS-WUP data (1975-2100)

In [ ]:
# List all GHS-WUP tif files
ghs_wup_files = sorted(list(ghs_wup_dir.glob("GHS_WUP_POP_E*_GLOBE_R2023A_4326_30ss_V1_0.tif")))
print(f"Found {len(ghs_wup_files)} GHS-WUP files:")
for f in ghs_wup_files:
    print(f"  {f.name}")

In [ ]:
# Read all GHS-WUP files and extract years
datasets_wup = []
years_wup = []

for tif_file in ghs_wup_files:
    # Read using rioxarray
    ds = rxr.open_rasterio(tif_file)
    
    # Extract year from filename (looking for pattern E{YYYY})
    year_match = re.search(r'E(\d{4})', tif_file.name)
    if year_match:
        year = int(year_match.group(1))
        years_wup.append(year)
        print(f"Processing year: {year}")
    else:
        print(f"Warning: Could not extract year from {tif_file.name}")
        continue
    
    datasets_wup.append(ds)

# Sort datasets and years together by year
sorted_pairs = sorted(zip(years_wup, datasets_wup), key=lambda x: x[0])
years_wup, datasets_wup = zip(*sorted_pairs)

# Concatenate along new time dimension
wup_ds = xr.concat(datasets_wup, dim="time")
wup_ds = wup_ds.rename("population")
wup_ds = wup_ds.squeeze("band", drop=True)

# Assign time based on extracted years
wup_ds["time"] = pd.to_datetime([f"{year}-01-01" for year in years_wup])
wup_ds = wup_ds.assign_coords(time=wup_ds.time)

# Replace potential fill values with NaN
wup_ds = wup_ds.where(wup_ds >= 0)

print(f"\nGHS-WUP dataset shape: {wup_ds.shape}")
print(f"Years available: {years_wup}")
wup_ds

## Create region masks and calculate population

In [ ]:
# Get bounding box for region
lon_min, lat_min, lon_max, lat_max = sel_gdf.geometry.total_bounds

# Subset GHS-POP to region bounding box
pop_region = pop_ds.sel(
    x=slice(lon_min, lon_max),
    y=slice(lat_max, lat_min)
)

# Create admin region mask for GHS-POP
pop_admin_mask_raw = admin_mask.mask(pop_region.x, pop_region.y)
pop_admin_mask = xr.DataArray(
    (~np.isnan(pop_admin_mask_raw.values)).astype(float),
    coords={'y': pop_region.y, 'x': pop_region.x},
    dims=['y', 'x']
)

print(f"GHS-POP admin mask: {pop_admin_mask.sum().values:.0f} grid cells")

In [ ]:
# Subset GHS-WUP to region bounding box
wup_region = wup_ds.sel(
    x=slice(lon_min, lon_max),
    y=slice(lat_max, lat_min)
)

# Create admin region mask for GHS-WUP
wup_admin_mask_raw = admin_mask.mask(wup_region.x, wup_region.y)
wup_admin_mask = xr.DataArray(
    (~np.isnan(wup_admin_mask_raw.values)).astype(float),
    coords={'y': wup_region.y, 'x': wup_region.x},
    dims=['y', 'x']
)

print(f"GHS-WUP admin mask: {wup_admin_mask.sum().values:.0f} grid cells")

## Visualize population distribution

In [ ]:
# Plot GHS-POP for selected years
selected_years = [0, len(years_pop)//2, -1]  # First, middle, last
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, time_idx in enumerate(selected_years):
    ax = axes[idx]
    year = pop_region.time.dt.year.values[time_idx]
    
    # Apply mask and plot
    masked_pop = pop_region.isel(time=time_idx).where(pop_admin_mask == 1)
    
    im = masked_pop.plot(
        ax=ax,
        cmap='YlOrRd',
        add_colorbar=True,
        cbar_kwargs={'label': 'Population per cell'},
        vmin=0
    )
    
    ax.set_title(f'Population {year} (GHS-POP)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    
    # Add region boundary
    sel_gdf.boundary.plot(ax=ax, color='black', linewidth=2)

plt.tight_layout()
plt.show()

## Calculate total population for each year

We calculate the total population within the region by summing all population values within the masked area.

In [ ]:
# Calculate total population for GHS-POP
pop_results = []

for i, year in enumerate(years_pop):
    # Get data for this year
    data = pop_region.isel(time=i)
    
    # Apply mask and sum population
    masked_data = data.where(pop_admin_mask == 1, 0)
    total_pop = masked_data.sum().values
    
    pop_results.append({
        'year': year,
        'population': total_pop
    })
    
    print(f"{year}: {total_pop:,.0f} people")

df_pop = pd.DataFrame(pop_results)
print(f"\nGHS-POP results:")
print(df_pop)

In [ ]:
# Calculate total population for GHS-WUP
wup_results = []

for i, year in enumerate(years_wup):
    # Get data for this year
    data = wup_region.isel(time=i)
    
    # Apply mask and sum population
    masked_data = data.where(wup_admin_mask == 1, 0)
    total_pop = masked_data.sum().values
    
    wup_results.append({
        'year': year,
        'population': total_pop
    })
    
    print(f"{year}: {total_pop:,.0f} people")

df_wup = pd.DataFrame(wup_results)
print(f"\nGHS-WUP results:")
print(df_wup)

## Save results to CSV

In [ ]:
# Create output directory
output_dir = data_dir / admin_id / "ghs_population"
output_dir.mkdir(parents=True, exist_ok=True)

# Save GHS-POP results
csv_file_pop = output_dir / f"population_ghs_pop_{admin_id}.csv"
df_pop.to_csv(csv_file_pop, index=False)
print(f"Saved GHS-POP results to: {csv_file_pop}")

# Save GHS-WUP results
csv_file_wup = output_dir / f"population_ghs_wup_{admin_id}.csv"
df_wup.to_csv(csv_file_wup, index=False)
print(f"Saved GHS-WUP results to: {csv_file_wup}")

## Visualize population timeseries

In [ ]:
# Plot both datasets together
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(df_pop['year'], df_pop['population'], 
        marker='o', linewidth=2, markersize=8, label='GHS-POP (1975-2030)', color='#2E86AB')
ax.plot(df_wup['year'], df_wup['population'], 
        marker='s', linewidth=2, markersize=8, label='GHS-WUP (1975-2100)', color='#A23B72', alpha=0.7)

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Population', fontsize=12)
ax.set_title(f'Population Change in {admin_id} (Central Greece)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Format y-axis to show numbers in thousands/millions
ax.ticklabel_format(style='plain', axis='y')
from matplotlib.ticker import FuncFormatter
def millions(x, pos):
    return f'{x/1e6:.1f}M' if x >= 1e6 else f'{x/1e3:.0f}K'
ax.yaxis.set_major_formatter(FuncFormatter(millions))

plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics
print("GHS-POP Summary (1975-2030):")
print(f"  Initial population (1975): {df_pop['population'].iloc[0]:,.0f}")
print(f"  Final population (2030): {df_pop['population'].iloc[-1]:,.0f}")
print(f"  Change: {df_pop['population'].iloc[-1] - df_pop['population'].iloc[0]:+,.0f} ({((df_pop['population'].iloc[-1] / df_pop['population'].iloc[0]) - 1) * 100:+.1f}%)")
print(f"  Peak population: {df_pop['population'].max():,.0f} in {df_pop.loc[df_pop['population'].idxmax(), 'year']:.0f}")

print("\nGHS-WUP Summary (1975-2100):")
print(f"  Initial population (1975): {df_wup['population'].iloc[0]:,.0f}")
print(f"  Final population (2100): {df_wup['population'].iloc[-1]:,.0f}")
print(f"  Change: {df_wup['population'].iloc[-1] - df_wup['population'].iloc[0]:+,.0f} ({((df_wup['population'].iloc[-1] / df_wup['population'].iloc[0]) - 1) * 100:+.1f}%)")
print(f"  Peak population: {df_wup['population'].max():,.0f} in {df_wup.loc[df_wup['population'].idxmax(), 'year']:.0f}")

## What's next?

You now have timeseries of population data that can be used as exposure in drought risk assessments. The CSV files are saved and ready to be used in other analyses.